# TCN Improved Training

This notebook retrains the TCN with two targeted improvements over `03_tcn_training.ipynb`:

| Change | Original | Improved |
|--------|----------|----------|
| LR Scheduler | `CosineAnnealingWarmRestarts` (T_0=10, T_mult=2) | `ReduceLROnPlateau` (factor=0.5, patience=5) |
| Max epochs | 50 | 60 |
| Early-stop patience | 10 | 15 |
| Threshold | Fixed 0.5 | Swept on val set → optimal |

**Why these changes?**
- `CosineAnnealingWarmRestarts` forcibly resets LR every 10/20/40 epochs, kicking the model out of the good minimum it found. This caused val F1 to swing wildly (0.73 → 0.63 → 0.54) in the original run.
- `ReduceLROnPlateau` only reduces LR when val F1 genuinely stalls — it never raises it back up.
- With 16.67% seizure class rate, threshold 0.5 is almost never optimal. The sweep finds the threshold that maximises val F1.

**Checkpoints saved to:** `checkpoints/TCN_improved_best.pt`  
**Results saved to:** `results/tcn_improved/`

---
## 1. Imports & Setup

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import warnings; warnings.filterwarnings('ignore')

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json, time, yaml
from pathlib import Path

from src.models import TCNClassifier, FocalLoss
from src.eval.trainer import run_epoch
from src.data.dataset import build_split_dataset, EEGDataset, build_train_loader, build_eval_loader

DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CKPT_DIR    = Path('checkpoints'); CKPT_DIR.mkdir(exist_ok=True)
RESULTS_DIR = Path('results/tcn_improved'); RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')
print(f'Results: {RESULTS_DIR}')

Device : cuda
PyTorch: 2.10.0+cu128
Results: results/tcn_improved


---
## 2. Data Loaders
Reuses the same cached `/tmp/neuroscribe_cache/` splits as the original notebook — no extra preprocessing cost.

In [2]:
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

_raw  = cfg['data']['raw_dir']
_ch   = cfg['data']['channels']
_wsec = cfg['data']['window_size']
_fs   = cfg['data']['sample_rate']
_ov   = cfg['data']['overlap']
_szt  = cfg['data']['seizure_threshold']
_bpl  = cfg['preprocessing']['bandpass_low']
_bph  = cfg['preprocessing']['bandpass_high']
_ntch = cfg['preprocessing']['notch_freq']
_seed = cfg['training']['seed']
_bs   = cfg['training']['batch_size']

CACHE_DIR = '/tmp/neuroscribe_cache'
os.makedirs(CACHE_DIR, exist_ok=True)


def _load_split(split, pids):
    cache_path = os.path.join(CACHE_DIR, f'{split}_subsampled.npz')
    if os.path.exists(cache_path):
        print(f'[{split}] Loading from /tmp cache...')
        d = np.load(cache_path)
        return EEGDataset(d['windows'], d['labels'], d['patient_ids'])

    print(f'[{split}] Building from raw EDF (first run only)...')
    ds = build_split_dataset(
        raw_dir=_raw, patient_ids=pids, split_name=split,
        target_channels=_ch, window_size_sec=_wsec, overlap=_ov,
        seizure_threshold=_szt, bandpass_low=_bpl, bandpass_high=_bph,
        notch_freq=_ntch, sample_rate=_fs,
        processed_dir=None, use_cache=False,
    )
    np.random.seed(_seed)
    s_idx  = np.where(ds.labels == 1)[0]
    ns_all = np.where(ds.labels == 0)[0]
    ns_idx = np.random.choice(ns_all, min(len(s_idx) * 5, len(ns_all)), replace=False)
    idx    = np.random.permutation(np.concatenate([s_idx, ns_idx]))
    ds     = EEGDataset(ds.windows[idx], ds.labels[idx], ds.patient_ids[idx])
    np.savez_compressed(cache_path, windows=ds.windows, labels=ds.labels, patient_ids=ds.patient_ids)
    print(f'  Cached → {cache_path}')
    return ds


train_ds = _load_split('train', cfg['splits']['train_patients'])
val_ds   = _load_split('val',   cfg['splits']['val_patients'])
test_ds  = _load_split('test',  cfg['splits']['test_patients'])

train_loader = build_train_loader(train_ds, batch_size=_bs, seed=_seed)
val_loader   = build_eval_loader(val_ds,   batch_size=_bs * 2)
test_loader  = build_eval_loader(test_ds,  batch_size=_bs * 2)

print(f'  Train : {len(train_ds):,}  seizure: {train_ds.n_seizure:,} ({train_ds.seizure_fraction:.2%})')
print(f'  Val   : {len(val_ds):,}  seizure: {val_ds.n_seizure:,} ({val_ds.seizure_fraction:.2%})')
print(f'  Test  : {len(test_ds):,}  seizure: {test_ds.n_seizure:,} ({test_ds.seizure_fraction:.2%})')

[train] Loading from /tmp cache...
[val] Loading from /tmp cache...
[test] Loading from /tmp cache...
  Train : 24,378  seizure: 4,063 (16.67%)
  Val   : 2,226  seizure: 371 (16.67%)
  Test  : 3,132  seizure: 522 (16.67%)


---
## 3. Model — Same Architecture as Original
Architecture is unchanged. Only the training schedule and threshold strategy differ.

In [3]:
tcn_imp = TCNClassifier(
    n_channels=23,
    proj_channels=64,
    num_filters=128,
    kernel_size=3,
    num_blocks=8,
    dropout=0.5,
)

params = sum(p.numel() for p in tcn_imp.parameters() if p.requires_grad)
print(f'TCN parameters  : {params:,}')
print(f'Receptive field : {tcn_imp.receptive_field} samples ({tcn_imp.receptive_field/256:.2f}s at 256Hz)')

TCN parameters  : 792,385
Receptive field : 1021 samples (3.99s at 256Hz)


---
## 4. Improved Training Loop

**Key change — `ReduceLROnPlateau` instead of `CosineAnnealingWarmRestarts`:**
```
Original:  LR resets to 3e-4 every 10/20/40 epochs → val F1 wildly unstable
Improved:  LR halved only when val F1 stalls for 5 epochs → smooth convergence
```

In [6]:
def add_noise(x, sigma=0.05):
    return x + sigma * torch.randn_like(x)


def train_tcn_improved(model, train_loader, val_loader, device,
                       lr=3e-4, weight_decay=5e-4, max_epochs=60,
                       patience=15, noise_sigma=0.05):

    ckpt_path = CKPT_DIR / 'TCN_improved_best.pt'
    log_path  = CKPT_DIR / 'TCN_improved_log.json'

    model = model.to(device)

    if ckpt_path.exists() and log_path.exists():
        print('[TCN-Improved] Loading saved checkpoint.')
        model.load_state_dict(torch.load(ckpt_path, map_location=device))
        return json.load(open(log_path))

    criterion = FocalLoss(alpha=0.85, gamma=2.0)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    # ── ReduceLROnPlateau ─────────────────────────────────────────────
    # Monitors val F1 (mode='max'). Halves LR if no improvement for
    # 5 epochs. Never resets LR upward — training stays in good minima.
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5,
        min_lr=1e-6,
    )

    history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': [],
               'best_epoch': 0, 'lr': []}
    best_val_f1, no_improve = 0.0, 0

    print(f'[TCN-Improved] Training up to {max_epochs} epochs  '
          f'(lr={lr}, wd={weight_decay}, noise_sigma={noise_sigma})...')
    print(f'  Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)')
    print(f'  Early stop patience: {patience}\n')

    for epoch in range(1, max_epochs + 1):
        t0 = time.time()

        model.train()
        total_loss, all_probs, all_labels = 0.0, [], []
        for X_batch, y_batch in train_loader:
            X_batch = add_noise(X_batch.to(device), sigma=noise_sigma)
            y_batch = y_batch.to(device)
            logits  = model(X_batch)
            loss    = criterion(logits, y_batch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
            all_probs.extend(torch.sigmoid(logits).detach().cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

        probs  = np.array(all_probs)
        labels = np.array(all_labels)
        preds  = (probs >= 0.5).astype(int)
        tp = int(((preds == 1) & (labels == 1)).sum())
        fp = int(((preds == 1) & (labels == 0)).sum())
        fn = int(((preds == 0) & (labels == 1)).sum())
        sens = tp / max(tp + fn, 1)
        prec = tp / max(tp + fp, 1)
        tr = {'loss': total_loss / len(labels),
              'f1':   2 * sens * prec / max(sens + prec, 1e-8)}

        vl = run_epoch(model, val_loader, criterion, None, device, threshold=0.5)

        # Step on val F1 — LR only decreases, never resets
        scheduler.step(vl['f1'])

        cur_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(tr['loss']); history['train_f1'].append(tr['f1'])
        history['val_loss'].append(vl['loss']);   history['val_f1'].append(vl['f1'])
        history['lr'].append(cur_lr)

        print(f'  Ep {epoch:02d}/{max_epochs}  '
              f'train loss={tr["loss"]:.4f} f1={tr["f1"]:.4f}  '
              f'val loss={vl["loss"]:.4f} f1={vl["f1"]:.4f}  '
              f'lr={cur_lr:.2e}  ({time.time()-t0:.0f}s)')

        if vl['f1'] > best_val_f1:
            best_val_f1 = vl['f1']
            history['best_epoch'] = epoch
            torch.save(model.state_dict(), ckpt_path)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stop (best epoch {history["best_epoch"]})')
                break

    json.dump(history, open(log_path, 'w'), indent=2)
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f'\nBest Val F1={best_val_f1:.4f} at epoch {history["best_epoch"]} → {ckpt_path.name}')
    return history

In [7]:
print('TCN-Improved Training\n' + '='*55)
hist = train_tcn_improved(tcn_imp, train_loader, val_loader, DEVICE)

TCN-Improved Training
[TCN-Improved] Training up to 60 epochs  (lr=0.0003, wd=0.0005, noise_sigma=0.05)...
  Scheduler: ReduceLROnPlateau (factor=0.5, patience=5)
  Early stop patience: 15

  Ep 01/60  train loss=0.0596 f1=0.6674  val loss=0.1706 f1=0.2857  lr=3.00e-04  (123s)
  Ep 02/60  train loss=0.0379 f1=0.7549  val loss=0.4494 f1=0.2857  lr=3.00e-04  (112s)
  Ep 03/60  train loss=0.0293 f1=0.8337  val loss=0.3297 f1=0.2859  lr=3.00e-04  (111s)
  Ep 04/60  train loss=0.0255 f1=0.8612  val loss=0.2368 f1=0.2872  lr=3.00e-04  (112s)
  Ep 05/60  train loss=0.0219 f1=0.8849  val loss=0.2115 f1=0.2873  lr=3.00e-04  (111s)
  Ep 06/60  train loss=0.0205 f1=0.8928  val loss=0.1016 f1=0.3368  lr=3.00e-04  (111s)
  Ep 07/60  train loss=0.0188 f1=0.9009  val loss=0.2621 f1=0.2859  lr=3.00e-04  (112s)
  Ep 08/60  train loss=0.0171 f1=0.9103  val loss=0.0535 f1=0.4164  lr=3.00e-04  (112s)
  Ep 09/60  train loss=0.0166 f1=0.9110  val loss=0.1019 f1=0.3329  lr=3.00e-04  (109s)
  Ep 10/60  train 

---
## 5. Training Curves

In [ ]:
epochs = range(1, len(hist['train_loss']) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, hist['train_loss'], '--', color='#8e44ad', alpha=0.7, label='Train')
axes[0].plot(epochs, hist['val_loss'],   '-',  color='#8e44ad',            label='Val')
axes[0].axvline(hist['best_epoch'], color='red', linestyle=':', label=f'Best ep {hist["best_epoch"]}')
axes[0].set_title('TCN-Improved — Focal Loss', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs, hist['train_f1'], '--', color='#8e44ad', alpha=0.7, label='Train')
axes[1].plot(epochs, hist['val_f1'],   '-',  color='#8e44ad',            label='Val')
axes[1].axvline(hist['best_epoch'], color='red', linestyle=':')
axes[1].set_title(f'TCN-Improved — F1 (best val={max(hist["val_f1"]):.4f})', fontweight='bold')
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs, hist['lr'], color='steelblue')
axes[2].set_title('LR — ReduceLROnPlateau', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('LR'); axes[2].grid(alpha=0.3)

plt.suptitle('TCN-Improved Training Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'tcn_improved_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {RESULTS_DIR}/tcn_improved_training_curves.png')

---
## 6. Threshold Optimisation on Validation Set
With 16.67% seizure class rate, threshold 0.5 is almost never optimal.  
We sweep 0.20–0.70 and pick the threshold that maximises val F1.

In [ ]:
# ── Collect val probabilities from best checkpoint ────────────────────────────
tcn_imp.eval()
val_probs, val_labels = [], []

with torch.no_grad():
    for X_batch, y_batch in val_loader:
        logits = tcn_imp(X_batch.to(DEVICE))
        val_probs.extend(torch.sigmoid(logits).cpu().numpy())
        val_labels.extend(y_batch.numpy())

val_probs  = np.array(val_probs)
val_labels = np.array(val_labels)

# ── Sweep thresholds ──────────────────────────────────────────────────────────
rows = []
for thr in np.arange(0.20, 0.75, 0.05):
    preds = (val_probs >= thr).astype(int)
    tp = int(((preds == 1) & (val_labels == 1)).sum())
    fp = int(((preds == 1) & (val_labels == 0)).sum())
    fn = int(((preds == 0) & (val_labels == 1)).sum())
    sens = tp / max(tp + fn, 1)
    prec = tp / max(tp + fp, 1)
    f1   = 2 * sens * prec / max(sens + prec, 1e-8)
    rows.append({'Threshold': round(thr, 2), 'Precision': round(prec, 4),
                 'Recall': round(sens, 4), 'F1': round(f1, 4),
                 'TP': tp, 'FP': fp, 'FN': fn})

df_thresh      = pd.DataFrame(rows)
best_row       = df_thresh.loc[df_thresh['F1'].idxmax()]
BEST_THRESHOLD = float(best_row['Threshold'])

print('Threshold sweep — validation set:')
print(df_thresh.to_string(index=False))
print()
print(f'  Best threshold : {BEST_THRESHOLD}')
print(f'  Val F1         : {best_row["F1"]:.4f}')
print(f'  Precision      : {best_row["Precision"]:.4f}')
print(f'  Recall         : {best_row["Recall"]:.4f}')

In [ ]:
# ── Plot threshold sweep ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_thresh['Threshold'], df_thresh['F1'],        marker='o', label='F1',        color='steelblue',      lw=2)
ax.plot(df_thresh['Threshold'], df_thresh['Precision'], marker='s', label='Precision', color='mediumseagreen', lw=1.5, alpha=0.8)
ax.plot(df_thresh['Threshold'], df_thresh['Recall'],    marker='^', label='Recall',    color='tomato',         lw=1.5, alpha=0.8)
ax.axvline(BEST_THRESHOLD, color='black', linestyle='--', lw=1.5, label=f'Best = {BEST_THRESHOLD}')
ax.axvline(0.5, color='grey', linestyle=':', lw=1, label='Default = 0.50')
ax.set_xlabel('Threshold'); ax.set_ylabel('Score')
ax.set_title('Threshold Optimisation on Validation Set', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {RESULTS_DIR}/threshold_sweep.png')

---
## 7. Summary

In [ ]:
gap = hist['train_f1'][hist['best_epoch'] - 1] - max(hist['val_f1'])

summary = {
    'total_epochs':          len(hist['train_loss']),
    'best_epoch':            hist['best_epoch'],
    'best_val_f1_thresh0.5': round(max(hist['val_f1']), 4),
    'best_threshold':        BEST_THRESHOLD,
    'val_f1_at_best_thresh': float(best_row['F1']),
    'val_precision':         float(best_row['Precision']),
    'val_recall':            float(best_row['Recall']),
    'final_train_f1':        round(hist['train_f1'][-1], 4),
    'train_val_gap':         round(gap, 4),
    'scheduler':             'ReduceLROnPlateau(factor=0.5, patience=5)',
}

with open(RESULTS_DIR / 'tcn_improved_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'Saved → {RESULTS_DIR}/tcn_improved_summary.json')
print(f'Checkpoint → checkpoints/TCN_improved_best.pt')

print()
print('='*55)
print('TCN-IMPROVED SUMMARY')
print('='*55)
print(f'  Total epochs trained  : {summary["total_epochs"]}')
print(f'  Best epoch            : {summary["best_epoch"]}')
print(f'  Val F1  (thresh=0.50) : {summary["best_val_f1_thresh0.5"]:.4f}')
print(f'  Val F1  (best thresh) : {summary["val_f1_at_best_thresh"]:.4f}  ← use this in testing')
print(f'  Best threshold        : {summary["best_threshold"]}')
print(f'  Precision             : {summary["val_precision"]:.4f}')
print(f'  Recall                : {summary["val_recall"]:.4f}')
print(f'  Train-val gap         : {summary["train_val_gap"]:.4f}')
print('='*55)
print()
print('To use this model in notebook 04_testing_comparison.ipynb:')
print(f'  ckpt_path  = "checkpoints/TCN_improved_best.pt"')
print(f'  threshold  = {BEST_THRESHOLD}')